# Week 07: Solutions

## Class `Backpack`

The class below is the test object that was availabel to import from `data_structures_lgreco`. The object fulfills the `Backpack` contract and works fine with any battery of tests that reflect that contract. The testing code is shown after the class `Backpack` below.

In [ ]:
from __future__ import annotations


class Backpack:

    __DEFAULT_CAPACITY = 5

    def __init__(self, owner: str, capacity: int = __DEFAULT_CAPACITY) -> None:
        """Initialize a Backpack with an owner and an optional capacity."""
        self.__owner: str = owner
        self.__capacity: int = capacity
        self.__items: list[str] = []

    def add_item(self, item: str) -> bool:
        """Add an item to the backpack if there is space available."""
        space_available = (len(self.__items) < self.__capacity) and item is not None
        if space_available:
            self.__items.append(item)
        return space_available

    def remove_item(self, item: str) -> bool:
        """Remove an item from the backpack if it exists."""
        found: bool = False
        i: int = 0
        if item is not None:
            while i < len(self.__items) and not found:
                found = self.__items[i] == item
                if found:
                    self.__items.pop(i)
                i += 1
        return found

    def count(self) -> int:
        """Return the number of items in the backpack."""
        return len(self.__items)

    def is_full(self) -> bool:
        """Return True if the backpack is full, False otherwise."""
        return self.count() == self.__capacity

    def items(self) -> list[str]:
        """Return a copy of the list of items in the backpack."""
        # Return a copy of the items list to prevent external modification of 
        # the backpack's internal state. In more professonal code, we might use 
        # `copy` module or list slicing to create a copy, but here we will 
        # do it manually to demonstrate the concept.
        items_list = []
        for item in self.__items:
            items_list.append(item)
        return items_list

    # Constants for string representation of the backpack
    __EMPTY = "empty"
    __OWNER = "Backpack(owner={}, "
    __ITEMS = "items={})"
    __RATIO = "{}/{}"

    def __str__(self) -> str:
        """Return a string representation of the backpack."""
        string_owner: str = self.__OWNER.format(self.__owner)
        string_items: str = self.__ITEMS.format(self.__EMPTY)
        if len(self.__items) > 0:
            string_items = self.__ITEMS.format(
                self.__RATIO.format(len(self.__items), self.__capacity)
            )
        return string_owner + string_items



---

## Class `Test_Backpack`

This is a testing class similar to what you may have written. Your testing class does not have to be identical to the one below. Just similar in the following manner:

* Your class must inherit `unittest.TestCase`.

* You must test all the contract specifications:

  * Constructor with default capacity

  * New object is always empty (tricky part: what if you instantiate an object with 0 capacity? Is it empty upon instantiation or full because `count==capacity`?)

  * Adding items while there is room (accept additions as long as `count < capacity`), etc.

* Each test method should have a docstring that describes the behavior being tested. This is important for readability and maintainability of your tests. When a test fails, the docstring is the diagnostic message the program will print to help you understand what went wrong. If your test methods do not have docstrings, the diagnostic messages will be less informative, making it harder to identify and fix issues in your code.



In [4]:
from __future__ import annotations
import unittest



class Test_Backpack(unittest.TestCase):

    def setUp(self) -> None:
        # Most tests can share this default setup
        self.b = Backpack("Test", 4)

    # -----------------------------
    # add_item
    # -----------------------------

    def test_add_item_rejects_none(self):
        """Adding None should fail and not change the backpack state."""
        self.assertEqual(self.b.count(), 0)
        self.assertFalse(self.b.add_item(None))
        self.assertEqual(self.b.count(), 0)  # state unchanged

    def test_add_item_up_to_capacity(self):
        """Adding items up to capacity should succeed and count should reflect."""  
        for item in ["pencil", "book", "lunch", "gloves"]:
            self.assertTrue(self.b.add_item(item))
        self.assertEqual(self.b.count(), 4)
        self.assertTrue(self.b.is_full())

    def test_add_item_above_capacity_fails_and_state_stays_full(self):
        """Adding items above capacity should fail and not change the backpack state."""        
        for item in ["pencil", "book", "lunch", "gloves"]:
            self.assertTrue(self.b.add_item(item))

        before = self.b.count()
        self.assertFalse(self.b.add_item("pen"))
        self.assertEqual(self.b.count(), before)  # no change
        self.assertTrue(self.b.is_full())

    # -----------------------------
    # remove_item
    # -----------------------------

    def test_remove_item_from_empty_returns_false(self):
        """Removing from an empty backpack should fail and not change the backpack state."""
        self.assertEqual(self.b.count(), 0)
        self.assertFalse(self.b.remove_item("pencil"))
        self.assertEqual(self.b.count(), 0)

    def test_remove_item_reduces_count_by_one(self):
        """Removing an item should reduce the count by one."""
        for _ in range(4):
            self.assertTrue(self.b.add_item("pencil"))

        # remove 4 times, each should reduce count by 1
        for expected_count in [3, 2, 1, 0]:
            self.assertTrue(self.b.remove_item("pencil"))
            self.assertEqual(self.b.count(), expected_count)

        # now empty: removing again should fail
        self.assertFalse(self.b.remove_item("pencil"))
        self.assertEqual(self.b.count(), 0)

    def test_remove_item_only_removes_one_occurrence(self):
        """Removing an item should only remove one occurrence."""
        self.b.add_item("pencil")
        self.b.add_item("pencil")
        self.b.add_item("book")

        self.assertTrue(self.b.remove_item("pencil"))
        self.assertEqual(self.b.items(), ["pencil", "book"])  # one pencil remains

    # -----------------------------
    # count / is_full
    # -----------------------------

    def test_count_tracks_adds_and_failed_adds(self):
        """Count should track successful adds and failed adds."""
        self.assertEqual(self.b.count(), 0)

        for _ in range(4):
            self.assertTrue(self.b.add_item("pencil"))
        self.assertEqual(self.b.count(), 4)

        # these should fail and not change count
        self.assertFalse(self.b.add_item("extra"))
        self.assertFalse(self.b.add_item("extra"))
        self.assertEqual(self.b.count(), 4)

    def test_is_full_transitions(self):
        """is_full should transition from False to True at capacity."""
        self.assertFalse(self.b.is_full())
        for _ in range(3):
            self.b.add_item("x")
            self.assertFalse(self.b.is_full())
        self.b.add_item("x")
        self.assertTrue(self.b.is_full())

    # -----------------------------
    # items()
    # -----------------------------

    def test_items_returns_copy_not_alias(self):
        """items() should return a copy of the items list, not an alias."""
        for item in ["pencil", "book"]:
            self.assertTrue(self.b.add_item(item))

        snapshot = self.b.items()
        snapshot[0] = "HACKED"   # mutate the returned list

        # The backpack state should not change
        self.assertEqual(self.b.items(), ["pencil", "book"])


# ----- Run the tests
#
# If you test in a .PY file, uncomment TEST-LINE-1 and TEST-LINE-2 and
# comment out TEST-LINE-3 to run the tests.
#
#if __name__ == "__main__":  #   TEST-LINE-1
#    unittest.main()  #   TEST-LINE-2
#
# If you test in a Jupyter notebook, comment out TEST-LINE-1 and TEST-LINE-2
# and uncomment TEST-LINE-C to run the tests in the notebook.
#
unittest.main(argv=[''], exit=False)    #   TEST-LINE-3
#


.........
----------------------------------------------------------------------
Ran 9 tests in 0.015s

OK
